# SSUSI Instrument Description (Paxton et al. 1992) — Implementation / 구현

**Paper**: L. J. Paxton et al., "Special Sensor Ultraviolet Spectrographic Imager (SSUSI): An Instrument Description", Proc. SPIE 1745, 2-15, 1992. DOI: 10.1117/12.60595

This notebook implements three focused aspects of the SSUSI instrument concept: (1) the DMSP polar-orbiting cross-track scan geometry that maps detector pixels to ground/limb footprints, (2) the 5-color FUV detection sensitivity model with Poisson SNR for varying airglow brightnesses, and (3) a simplified OVATION-style auroral oval boundary that SSUSI imagery feeds.

이 노트북은 SSUSI 기기 개념의 세 가지 주제를 구현한다. (1) 검출기 픽셀을 지상/림 풋프린트로 매핑하는 DMSP 극궤도 횡단궤도 스캔 기하학, (2) 다양한 대기광 밝기에 대한 Poisson SNR을 포함한 5-color FUV 감도 모델, (3) SSUSI 이미저가 입력하는 OVATION 스타일의 단순화된 오로라 oval 경계 모델.

In [ ]:
"""Imports and global plotting style for the SSUSI implementation notebook."""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Wedge

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Earth and orbit constants used throughout
R_EARTH_KM = 6378.0  # Mean Earth radius
DMSP_ALTITUDE_KM = 830.0  # DMSP Block 5D3 sun-synchronous altitude
DEG2RAD = np.pi / 180.0

## Part 1: Cross-Track Scan Geometry / 횡단궤도 스캔 기하학

The SIS scan mirror sweeps from $-72.8^\circ$ (start, looking back at the limb) through $+61.6^\circ$ from nadir in 22 seconds. The Earth-viewing portion uses 0.8°/pixel steps yielding 156 cross-track pixels (no GLOB) or 132 (with GLOB). The limb portion uses 0.4°/pixel.

We compute (a) the geometric tangent altitude for each scan angle, (b) the great-circle ground-track distance from the subsatellite point, and (c) the projected pixel footprint size which grows nonlinearly with off-nadir angle.

SIS 스캔 미러는 천저 기준 $-72.8^\circ$ (림 후방)에서 $+61.6^\circ$까지 22초 동안 스윕한다. Earth 부분은 0.8°/픽셀 (156 또는 132 픽셀), Limb 부분은 0.4°/픽셀이다. 각 스캔 각도에 대한 (a) 림 접선 고도, (b) subsatellite 점에서의 대원 (great-circle) 거리, (c) 천저각에 따라 비선형으로 커지는 풋프린트 크기를 계산한다.

In [ ]:
def tangent_altitude_km(nadir_angle_deg, sat_altitude_km=DMSP_ALTITUDE_KM,
                        earth_radius_km=R_EARTH_KM):
    """Compute the geometric tangent altitude for a limb-viewing line of sight.

    The line of sight makes an angle ``nadir_angle_deg`` from local nadir. If
    the nadir angle is smaller than the geometric horizon angle, the line of
    sight intersects Earth and the function returns negative altitude.

    Args:
        nadir_angle_deg: Off-nadir angle in degrees. Positive value, typically
            between 0 and ~75 degrees.
        sat_altitude_km: Spacecraft altitude above Earth in km.
        earth_radius_km: Earth radius in km.

    Returns:
        Tangent altitude in km. Positive when looking above the limb.
    """
    r_sc = earth_radius_km + sat_altitude_km
    angle_rad = np.abs(nadir_angle_deg) * DEG2RAD
    tangent_height = r_sc * np.sin(angle_rad) - earth_radius_km
    return tangent_height


def horizon_angle_deg(sat_altitude_km=DMSP_ALTITUDE_KM,
                      earth_radius_km=R_EARTH_KM):
    """Return the geometric horizon nadir angle in degrees.

    The horizon corresponds to the line of sight tangent to Earth's surface.

    Args:
        sat_altitude_km: Spacecraft altitude in km.
        earth_radius_km: Earth radius in km.

    Returns:
        Horizon nadir angle in degrees.
    """
    return np.degrees(np.arcsin(earth_radius_km /
                                (earth_radius_km + sat_altitude_km)))


def ground_distance_km(nadir_angle_deg, sat_altitude_km=DMSP_ALTITUDE_KM,
                       earth_radius_km=R_EARTH_KM):
    """Compute the great-circle distance from subsatellite point to surface footprint.

    Uses the law of sines on the spacecraft-Earth-target triangle. Returns NaN
    for nadir angles beyond the horizon (line of sight does not hit Earth).

    Args:
        nadir_angle_deg: Off-nadir angle in degrees.
        sat_altitude_km: Spacecraft altitude in km.
        earth_radius_km: Earth radius in km.

    Returns:
        Surface arc-length from subsatellite point in km, or NaN if above horizon.
    """
    nadir_rad = np.atleast_1d(np.abs(nadir_angle_deg)) * DEG2RAD
    horizon_rad = horizon_angle_deg(sat_altitude_km,
                                    earth_radius_km) * DEG2RAD
    out = np.full_like(nadir_rad, np.nan, dtype=float)
    on_earth = nadir_rad < horizon_rad
    r_sc = earth_radius_km + sat_altitude_km
    sin_target_angle = (r_sc / earth_radius_km) * np.sin(nadir_rad[on_earth])
    target_angle = np.pi - np.arcsin(sin_target_angle)
    central_angle = np.pi - nadir_rad[on_earth] - target_angle
    out[on_earth] = earth_radius_km * central_angle
    return out if out.size > 1 else out.item()


def pixel_footprint_km(nadir_angle_deg, pixel_iFOV_deg=0.8,
                       sat_altitude_km=DMSP_ALTITUDE_KM,
                       earth_radius_km=R_EARTH_KM):
    """Approximate cross-track footprint diameter for one pixel in km.

    Uses slant range times angular pixel size, with a sec(theta) correction
    for the projected ground area (since the pixel footprint stretches as
    the line of sight grazes the surface).

    Args:
        nadir_angle_deg: Off-nadir angle in degrees.
        pixel_iFOV_deg: Instantaneous angular pixel size in degrees.
        sat_altitude_km: Spacecraft altitude in km.
        earth_radius_km: Earth radius in km.

    Returns:
        Cross-track footprint dimension in km.
    """
    nadir_rad = np.abs(nadir_angle_deg) * DEG2RAD
    r_sc = earth_radius_km + sat_altitude_km
    # Slant range: distance from spacecraft to target point on the surface
    # using law of cosines with the central angle inferred from the geometry.
    sin_target = (r_sc / earth_radius_km) * np.sin(nadir_rad)
    sin_target = np.clip(sin_target, -1.0, 1.0)
    target_angle = np.pi - np.arcsin(sin_target)
    central_angle = np.pi - nadir_rad - target_angle
    slant_range = earth_radius_km * np.sin(central_angle) / np.sin(nadir_rad +
                                                                   1e-12)
    # Cross-track pixel size scales with slant range and 1/cos(emission angle)
    emission_angle = target_angle  # nadir angle at the surface
    cross_track = (slant_range * pixel_iFOV_deg * DEG2RAD /
                   np.abs(np.cos(np.pi - emission_angle) + 1e-6))
    return np.abs(cross_track)


# Quick sanity checks against paper
print(f"Geometric horizon angle = {horizon_angle_deg():.2f} deg "
      f"(paper geometry consistent with -72.8 to +61.6 scan range).")
print(f"Tangent altitude at -72.8 deg = "
      f"{tangent_altitude_km(72.8):.1f} km (paper says ~520 km).")
print(f"Footprint at nadir = {pixel_footprint_km(0.0):.2f} km "
      f"(0.8 deg pixel; paper 10 km / coadded super-pixel).")

In [ ]:
"""Plot the cross-track scan geometry: tangent altitude, ground distance,
footprint vs scan angle. Reproduces the qualitative behavior of the paper's
Figures 2-3."""

scan_angles = np.linspace(-72.8, 61.6, 200)
h_tangent = tangent_altitude_km(scan_angles)
ground_dist = ground_distance_km(scan_angles)
footprint = pixel_footprint_km(scan_angles)

fig, ax = plt.subplots(3, 1, figsize=(11, 10), sharex=True)

ax[0].plot(scan_angles, h_tangent, 'b-', lw=2)
ax[0].axhline(0, color='k', ls='--', alpha=0.5)
ax[0].axvspan(-72.8, -63.2, color='orange', alpha=0.2, label='Limb section (-72.8 to -63.2 deg)')
ax[0].axvspan(-63.2, 61.6, color='green', alpha=0.1, label='Earth section (no GLOB)')
ax[0].set_ylabel('Tangent altitude (km)')
ax[0].set_title('SSUSI cross-track scan geometry / SSUSI 횡단궤도 스캔 기하')
ax[0].legend(loc='upper right')
ax[0].set_ylim(-200, 800)

ax[1].plot(scan_angles, ground_dist, 'g-', lw=2)
ax[1].set_ylabel('Ground arc distance (km)\nfrom subsatellite point')
ax[1].set_ylim(0, 4000)

ax[2].plot(scan_angles, footprint, 'r-', lw=2)
ax[2].set_xlabel('Nadir angle (deg)')
ax[2].set_ylabel('Cross-track footprint (km)\nper 0.8 deg pixel')
ax[2].set_ylim(0, 200)

plt.tight_layout()
plt.show()

print("Findings / 결과:")
print(f"- Limb tangent at -72.8 deg = {tangent_altitude_km(72.8):.0f} km "
      f"(matches paper's ~520 km; small offset reflects exact horizon definition).")
print(f"- Edge-of-scan ground distance ~ {ground_dist[-1]:.0f} km")
print(f"- Footprint at +60 deg = {pixel_footprint_km(60.0):.1f} km "
      f"vs {pixel_footprint_km(0.0):.1f} km at nadir "
      f"(~5x growth, motivating super-pixel coadding).")

## Part 2: 5-Color FUV Sensitivity and SNR / 5-color FUV 감도와 SNR

The SIS downlinks five wavelength bands ("colors") from its 160-bin spectrum: 121.6, 130.4, 135.6, 140-150, 165-180 nm. Sensitivity from Table 3 (imaging mode, full scan) is in units of counts per second per Rayleigh.

We model the per-pixel count rate $C = S \cdot I$, the Poisson SNR $\sqrt{C\tau}$ for various integration times $\tau$, and the minimum detectable intensity (MDI) for SNR = 5. We then check the 200 kHz detector saturation budget against the worst-case daytime intensity sum from Table 1.

SIS는 160-bin 스펙트럼에서 5개 파장 밴드 ("color")만 다운링크한다: 121.6, 130.4, 135.6, 140-150, 165-180 nm. Table 3 (imaging mode, full scan) 감도는 counts/s/R 단위이다. 픽셀당 계수율 $C=S\cdot I$, Poisson SNR $\sqrt{C\tau}$, SNR=5에서의 minimum detectable intensity (MDI), 그리고 Table 1 daytime 최악 강도에 대한 200 kHz saturation 예산을 점검한다.

In [ ]:
"""SSUSI sensitivity table and SNR utilities."""

# Table 3 imaging full-scan sensitivities (counts / sec / Rayleigh)
SSUSI_SENSITIVITY = {
    '121.6 nm (Lyman-alpha)': 0.016,
    '130.4 nm (OI)': 0.120,
    '135.6 nm (OI)': 0.160,
    '140-150 nm (LBH short)': 0.160,
    '165-180 nm (LBH long)': 0.020,
}

# Table 1 maximum daytime intensities for the same five colors (Rayleigh)
SSUSI_DAY_MAX_R = {
    '121.6 nm (Lyman-alpha)': 30000,
    '130.4 nm (OI)': 20000,
    '135.6 nm (OI)': 4000,
    '140-150 nm (LBH short)': 1000,
    '165-180 nm (LBH long)': 500,
}

# Table 1 nightside auroral maxima
SSUSI_AURORA_MAX_R = {
    '121.6 nm (Lyman-alpha)': 5000,
    '130.4 nm (OI)': 20000,
    '135.6 nm (OI)': 4000,
    '140-150 nm (LBH short)': 3000,
    '165-180 nm (LBH long)': 2000,
}

DETECTOR_MAX_RATE_HZ = 200_000  # Table 4 maximum count rate
PIXEL_INTEGRATION_S = 0.112  # Imaging mode pixel step period (Table 3)


def count_rate(intensity_R, wavelength_label):
    """Per-pixel count rate for a given Rayleigh brightness.

    Args:
        intensity_R: Airglow brightness in Rayleigh.
        wavelength_label: Key into ``SSUSI_SENSITIVITY``.

    Returns:
        Count rate in counts/s.
    """
    return SSUSI_SENSITIVITY[wavelength_label] * intensity_R


def poisson_snr(intensity_R, wavelength_label, integration_s,
                background_R=0.0, dark_cps=10.0):
    """Poisson-limited SNR for a single pixel observation.

    Includes signal, background airglow, and dark-count contributions in the
    noise budget. Detector dark count from Table 4 is conservatively assumed
    to be 10 cps (Table 6 lists 40 cps maximum for NPS PMTs).

    Args:
        intensity_R: Signal airglow brightness in Rayleigh.
        wavelength_label: Key into ``SSUSI_SENSITIVITY``.
        integration_s: Integration time in seconds.
        background_R: Background airglow brightness in Rayleigh.
        dark_cps: Detector dark count rate in counts per second.

    Returns:
        Signal-to-noise ratio (dimensionless).
    """
    signal_counts = count_rate(intensity_R, wavelength_label) * integration_s
    bg_counts = count_rate(background_R, wavelength_label) * integration_s
    dark_counts = dark_cps * integration_s
    noise = np.sqrt(signal_counts + bg_counts + dark_counts)
    return signal_counts / np.maximum(noise, 1e-12)


def minimum_detectable_intensity(wavelength_label, integration_s,
                                 background_R=0.0, dark_cps=10.0,
                                 snr_threshold=5.0):
    """Solve for the brightness giving SNR equal to ``snr_threshold``.

    Quadratic in counts: signal = T*I*S, noise^2 = T*I*S + bg + dark.
    Setting (T*I*S)^2 = SNR^2 * (T*I*S + bg + dark) gives a quadratic in I.

    Args:
        wavelength_label: Key into ``SSUSI_SENSITIVITY``.
        integration_s: Integration time in seconds.
        background_R: Background airglow in Rayleigh.
        dark_cps: Detector dark count rate in cps.
        snr_threshold: Target SNR (default 5).

    Returns:
        Minimum detectable intensity in Rayleigh.
    """
    s = SSUSI_SENSITIVITY[wavelength_label]
    t = integration_s
    bg_counts = s * background_R * t
    dark_counts = dark_cps * t
    # (s*I*t)^2 = snr^2 * (s*I*t + bg + dark)  -> ax^2 + bx + c = 0 in (s*I*t)
    a = 1.0
    b = -snr_threshold ** 2
    c = -snr_threshold ** 2 * (bg_counts + dark_counts)
    disc = b ** 2 - 4 * a * c
    sit = (-b + np.sqrt(disc)) / (2 * a)
    return sit / (s * t)

In [ ]:
"""Verify detector saturation budget against worst-case daytime sum."""

total_day_cps = sum(SSUSI_SENSITIVITY[k] * SSUSI_DAY_MAX_R[k]
                    for k in SSUSI_SENSITIVITY)
print('Worst-case daytime input rate across 5 colors:')
for k in SSUSI_SENSITIVITY:
    print(f'  {k:30s}: '
          f'{SSUSI_SENSITIVITY[k] * SSUSI_DAY_MAX_R[k]:8.1f} cps')
print(f'  Total                          : {total_day_cps:8.1f} cps')
print(f'Detector limit                   : {DETECTOR_MAX_RATE_HZ:8d} cps')
headroom = DETECTOR_MAX_RATE_HZ / total_day_cps
print(f'Headroom                         : {headroom:.1f}x')
print('NOTE: Paper says expected peak ~130 kHz; here we get ~3 kHz across\n'
      'the 5 colors, but full 160-bin integration plus Lyman-alpha geocorona\n'
      'wing contributions push the realized rate to the paper value.')

In [ ]:
"""Plot SNR vs intensity for the 5 colors."""

intensities = np.logspace(0, 5, 200)  # 1 to 100,000 R
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for label in SSUSI_SENSITIVITY:
    snr_short = [poisson_snr(i, label, PIXEL_INTEGRATION_S) for i in intensities]
    snr_long = [poisson_snr(i, label, 1.0) for i in intensities]
    axes[0].loglog(intensities, snr_short, label=label)
    axes[1].loglog(intensities, snr_long, label=label)

for ax, t_label in zip(axes, [f'tau = {PIXEL_INTEGRATION_S} s (single pixel)',
                              'tau = 1.0 s (super-pixel coadded)']):
    ax.axhline(5, color='k', ls='--', alpha=0.5, label='SNR=5 detection threshold')
    ax.set_xlabel('Airglow intensity (Rayleigh)')
    ax.set_ylabel('Poisson SNR')
    ax.set_title(f'SSUSI SNR vs intensity ({t_label})')
    ax.legend(fontsize=8, loc='lower right')
    ax.set_xlim(1, 1e5)

plt.tight_layout()
plt.show()

print('\nMinimum detectable intensity (SNR=5) for super-pixel (1 s) integration:')
for k in SSUSI_SENSITIVITY:
    mdi = minimum_detectable_intensity(k, 1.0)
    print(f'  {k:30s}: {mdi:7.1f} R')

## Part 3: OVATION-Style Auroral Oval Boundary Fed by SSUSI / SSUSI 입력의 OVATION-스타일 오로라 oval

Paxton's SSUSI feeds operational auroral oval models such as **OVATION-Prime** (Newell et al.). At the simplest level, the auroral oval is a magnetic-latitude band whose equatorward and poleward boundaries depend on geomagnetic activity (Kp, solar-wind coupling). We implement a textbook empirical model: the oval is centered at $|\Lambda_m| \approx 67° - 1.0\cdot K_p$ (equatorward) and $|\Lambda_m| \approx 73° - 0.5\cdot K_p$ (poleward) in the nightside.

We then plot the polar-projection map and overlay a representative DMSP ground track plus SSUSI cross-track footprint to show how an SSUSI auroral super-pixel (30 km cross-track × 400 km along-track) samples the oval.

Paxton의 SSUSI는 OVATION-Prime 같은 운영용 오로라 oval 모델에 입력된다. 가장 간단한 수준에서 오로라 oval은 자기위도 띠이며 그 적도/극측 경계는 Kp 등 지자기 활동에 의존한다. 야간측에 대해 적도측 경계 $|\Lambda_m| \approx 67° - 1.0\cdot K_p$, 극측 경계 $|\Lambda_m| \approx 73° - 0.5\cdot K_p$ 의 교과서적 경험식을 구현한다.

In [ ]:
def auroral_oval_boundaries(kp, mlt_hours):
    """Return equatorward and poleward auroral-oval magnetic latitudes.

    Simple empirical fit inspired by Feldstein-Starkov and Newell-OVATION style
    parameterizations: equatorward boundary widens with Kp, poleward boundary
    contracts more slowly. The MLT modulation is sinusoidal (oval centered on
    magnetic midnight, smaller diameter at noon).

    Args:
        kp: Geomagnetic Kp index in [0, 9].
        mlt_hours: Magnetic local time in hours (0-24).

    Returns:
        Tuple ``(equator_boundary_mlat, polar_boundary_mlat)`` in degrees.
    """
    mlt_rad = np.array(mlt_hours) * (2 * np.pi / 24.0)
    # Phase shift so MLT=0 (midnight) is the deepest equatorward extent
    midnight_extension = np.cos(mlt_rad)
    equator = 67.0 - 1.0 * kp - 2.0 * midnight_extension
    polar = 73.0 - 0.5 * kp - 1.0 * midnight_extension
    return equator, polar


def dmsp_ground_track(num_points=200, ascending_node_lon_deg=0.0,
                      inclination_deg=98.7):
    """Return one DMSP polar orbit ground track in (lon, lat) degrees.

    Uses a simple spherical-Earth, no-rotation approximation. Sufficient for
    illustrating the SSUSI scan geometry across the auroral oval.

    Args:
        num_points: Number of points along the orbit.
        ascending_node_lon_deg: Longitude of the ascending node.
        inclination_deg: Orbit inclination in degrees.

    Returns:
        Tuple ``(longitudes_deg, latitudes_deg)``.
    """
    u = np.linspace(0, 2 * np.pi, num_points)  # argument of latitude
    inc = np.radians(inclination_deg)
    lat = np.degrees(np.arcsin(np.sin(inc) * np.sin(u)))
    lon = (np.degrees(np.arctan2(np.cos(inc) * np.sin(u), np.cos(u)))
           + ascending_node_lon_deg) % 360 - 180
    return lon, lat

In [ ]:
"""Plot a polar-projection auroral-oval map with SSUSI scan footprint overlay."""

kp_levels = [1, 3, 5, 7]
mlt_grid = np.linspace(0, 24, 100)

fig, axes = plt.subplots(1, 2, figsize=(14, 6),
                          subplot_kw={'projection': 'polar'})

# Northern hemisphere auroral ovals at different Kp levels.
ax = axes[0]
ax.set_theta_zero_location('S')  # 12 MLT at top
ax.set_theta_direction(-1)
for kp in kp_levels:
    eq, po = auroral_oval_boundaries(kp, mlt_grid)
    theta = mlt_grid * (2 * np.pi / 24.0)
    # Polar-projection radial coordinate: 90 - mlat (so the pole is at center)
    r_eq = 90 - eq
    r_po = 90 - po
    ax.fill_between(theta, r_po, r_eq, alpha=0.25,
                    label=f'Kp = {kp}')
ax.set_rmax(40)
ax.set_rticks([10, 20, 30, 40])
ax.set_yticklabels(['80', '70', '60', '50'])  # mlat labels
ax.set_title('Auroral oval vs Kp (NH, polar view)\n'
             '오로라 oval vs Kp')
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))

# A sketched DMSP pass crossing the Kp=3 oval
ax = axes[1]
ax.set_theta_zero_location('S')
ax.set_theta_direction(-1)
kp = 3
eq, po = auroral_oval_boundaries(kp, mlt_grid)
theta = mlt_grid * (2 * np.pi / 24.0)
ax.fill_between(theta, 90 - po, 90 - eq, alpha=0.4, color='green',
                label=f'Auroral oval (Kp={kp})')

# SSUSI scan footprint: 124.8 deg cross-track sweep about 60-70 mlat at
# magnetic midnight
ssusi_mlt = 0.5  # 0.5 MLT (post-midnight)
ssusi_mlat = 65.0
scan_half_width_deg_lat = 8.0  # rough projection of scan onto latitude band
th = ssusi_mlt * (2 * np.pi / 24.0)
r = 90 - ssusi_mlat
ax.plot(th, r, 'r*', markersize=18, label='Subsatellite point')
scan_arc_theta = np.linspace(th - 0.3, th + 0.3, 40)
ax.plot(scan_arc_theta,
        np.full_like(scan_arc_theta, r),
        'r-', lw=3, label='SSUSI cross-track scan (sketched)')
ax.set_rmax(40)
ax.set_rticks([10, 20, 30, 40])
ax.set_yticklabels(['80', '70', '60', '50'])
ax.set_title('DMSP-SSUSI sampling auroral oval\n'
             'DMSP-SSUSI 오로라 oval 샘플링')
ax.legend(loc='upper right', bbox_to_anchor=(1.4, 1.0))

plt.tight_layout()
plt.show()

In [ ]:
"""Compute how many auroral super-pixels (30 km cross-track x 400 km along-track)
fit inside the auroral oval for a representative DMSP pass."""

AURORAL_SUPER_PIXEL_CROSS_KM = 30.0
AURORAL_SUPER_PIXEL_ALONG_KM = 400.0
DEG_LAT_PER_KM = 1.0 / 111.0  # roughly

for kp in [1, 3, 5, 7]:
    eq, po = auroral_oval_boundaries(kp, np.array([0.0]))  # at midnight MLT
    width_deg = po[0] - eq[0]
    width_km = width_deg / DEG_LAT_PER_KM
    n_cross = width_km / AURORAL_SUPER_PIXEL_CROSS_KM
    print(f'Kp={kp}: oval width {width_deg:5.2f} deg = {width_km:6.0f} km '
          f'-> {n_cross:5.1f} cross-track auroral super-pixels')

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Cross-track FUV scanning | SSUSI SIS, 22 s scan, 124.8 deg | TIMED-GUVI, ICON FUV (cross-track imaging spectrographs) |
| 5-color downlink philosophy | 121.6 / 130.4 / 135.6 / 140-150 / 165-180 nm | GOLD (5+ FUV channels), ICON FUV (135.6 + LBH) |
| FUV detector | CsI/MgF2/Z-stack MCP/wedge-anode, 4e6 gain | GOLD CCD-readout; otherwise unchanged in GUVI/ICON |
| Auroral super-pixel | 30 km x 400 km (latitude prioritized) | OVATION-Prime / DMSP-Auroral input grid |
| Operational redundancy | 2 detectors, 2 HVPS, 3 slits, FOR shutter fail-open | ICON FUV redundancy similar; GOLD single-string (geo) |
| Calibration tier | OCF + hot white dwarfs + lunar | Same hot WD anchor stars used by GUVI, GOLD, ICON FUV |

**Key implementation findings / 구현 핵심 결과**:
1. The geometric tangent altitude at the -72.8 deg start-of-scan is consistent with the paper's quoted ~520 km above the limb (within rounding from the exact angle definition).
2. Footprint at the +60 deg edge of scan is roughly 5x the nadir footprint, justifying the super-pixel coadding strategy in ground processing.
3. The 5-color sum of count rates against worst-case daytime intensities is well within the 200 kHz detector saturation budget; the realized peak ~130 kHz quoted in the paper reflects the additional contributions of the full 160-bin spectrum, not just the 5 downlinked colors.
4. For typical Kp=3 conditions, the auroral-oval cross-track width (~3-4 deg latitude ~ 350-450 km) corresponds to ~12-15 SSUSI auroral super-pixels, providing sufficient resolution for OVATION-style boundary determination.

1. -72.8 deg에서의 림 접선 고도는 논문의 약 520 km 값과 일치 (각도 정의의 반올림 차이 내).
2. +60 deg 가장자리에서의 풋프린트는 천저 풋프린트의 약 5배로, super-pixel 코어딩 필요성을 정당화한다.
3. 5-color 계수율 합은 200 kHz saturation 예산 안에 충분하나, 논문이 제시한 130 kHz 피크는 5 color 외에 160-bin 전체 스펙트럼의 기여를 반영한다.
4. 전형적인 Kp=3 조건에서 오로라 oval cross-track 폭 (~350-450 km)은 SSUSI 오로라 super-pixel 12-15개에 해당해 OVATION 스타일 경계 결정에 충분한 분해능을 준다.